In [0]:
dbutils.widgets.text("catalog_name",    "food_inspection")
dbutils.widgets.text("bronze_schema",   "bronze")
dbutils.widgets.text("silver_schema",   "silver")
dbutils.widgets.text("metadata_schema", "metadata")
 
catalog_name    = dbutils.widgets.get("catalog_name")
bronze_schema   = dbutils.widgets.get("bronze_schema")
silver_schema   = dbutils.widgets.get("silver_schema")
metadata_schema = dbutils.widgets.get("metadata_schema")
 
spark.sql(f"USE CATALOG {catalog_name}")
print(f"Silver pipeline — {catalog_name}")

In [0]:
from pyspark.sql.functions import (
    col, lit, trim, upper, lower, when, split, explode, posexplode,
    regexp_extract, regexp_replace, to_date, concat_ws, md5, length,
    current_timestamp, current_date, coalesce, size, count as spark_count,
    row_number, monotonically_increasing_id, max as spark_max
)
from pyspark.sql.window import Window
from pyspark.sql import DataFrame
from functools import reduce
from datetime import datetime
 
def log_dqx(table_name, city, total, passed, failed):
    spark.sql(f"""
        INSERT INTO {catalog_name}.{metadata_schema}.dqx_execution_log VALUES (
            '{table_name}', '{city}', current_timestamp(),
            {total}, {passed}, {failed}, current_date()
        )
    """)

In [0]:
#read bronze chicago
chicago_bronze = spark.table(f"{catalog_name}.{bronze_schema}.bronze_chicago")
chi_source_count = chicago_bronze.count()
print(f"Bronze Chicago rows: {chi_source_count}")

In [0]:
#renaming columns to snake_case
chicago = (chicago_bronze
    .withColumnRenamed("Inspection_ID",   "inspection_id")
    .withColumnRenamed("DBA_Name",        "dba_name")
    .withColumnRenamed("AKA_Name",        "aka_name")
    .withColumnRenamed("License_num",     "license_number")
    .withColumnRenamed("Facility_Type",   "facility_type")
    .withColumnRenamed("Risk",            "risk")
    .withColumnRenamed("Address",         "address")
    .withColumnRenamed("City",            "city")
    .withColumnRenamed("State",           "state")
    .withColumnRenamed("Zip",             "zip_code")
    .withColumnRenamed("Inspection_Date", "inspection_date_raw")
    .withColumnRenamed("Inspection_Type", "inspection_type")
    .withColumnRenamed("Results",         "results")
    .withColumnRenamed("Violations",      "violations_raw")
    .withColumnRenamed("Latitude",        "latitude_raw")
    .withColumnRenamed("Longitude",       "longitude_raw")
    .withColumnRenamed("Location",        "location_raw")
)
 
print(f"Columns renamed to snake_case")

In [0]:
#type casting
chicago = (chicago
    .withColumn("inspection_id",    col("inspection_id").cast("string"))
    .withColumn("license_number",   col("license_number").cast("string"))
    .withColumn("zip_code",         trim(col("zip_code")).cast("string"))
    .withColumn("inspection_date",  to_date(col("inspection_date_raw"), "MM/dd/yyyy"))
    .withColumn("latitude",         col("latitude_raw").cast("double"))
    .withColumn("longitude",        col("longitude_raw").cast("double"))
    .drop("inspection_date_raw", "latitude_raw", "longitude_raw", "location_raw")
)
 
print("Types cast: inspection_date→DATE, lat/long→DOUBLE, zip/license→STRING")

In [0]:
#trimming white spaces on string columns
string_cols = ["dba_name", "aka_name", "facility_type", "risk", 
               "address", "city", "state", "inspection_type", "results"]
 
for c in string_cols:
    chicago = chicago.withColumn(c, trim(col(c)))
 
print(f"Trimmed {len(string_cols)} string columns")
 

In [0]:
#treating license number = 0 as NULL
chi_zero_license = chicago.filter(col("license_number") == "0").count()
print(f"License number = 0 (treating as NULL): {chi_zero_license}")
 
chicago = chicago.withColumn("license_number",
    when(col("license_number") == "0", lit(None)).otherwise(col("license_number"))
)

In [0]:
#derive violation score from results
chicago = chicago.withColumn("violation_score",
    when(col("results") == "Pass", 90)
    .when(col("results") == "Pass w/ Conditions", 80)
    .when(col("results") == "Fail", 70)
    .when(col("results") == "No Entry", 0)
    .otherwise(lit(None).cast("int"))
)
 
print("Derived violation_score from Results")
display(chicago.groupBy("results", "violation_score").count().orderBy("violation_score"))

In [0]:
 #flag out of state records 
chi_out_of_state = chicago.filter(col("state") != "IL").count()
print(f"Out-of-state records (non-IL): {chi_out_of_state}")
 
chicago = chicago.withColumn("is_out_of_area",
    when(col("state") != "IL", lit(True)).otherwise(lit(False))
)

In [0]:
# Build quarantine reason column
chicago = chicago.withColumn("_quarantine_reason", lit(None).cast("string"))
 
# VR-001: dba_name cannot be null/empty
chicago = chicago.withColumn("_quarantine_reason",
    when(col("dba_name").isNull() | (trim(col("dba_name")) == ""), lit("dba_name_null"))
    .otherwise(col("_quarantine_reason"))
)
 
# VR-002: inspection_date cannot be null
chicago = chicago.withColumn("_quarantine_reason",
    when(col("inspection_date").isNull(),
        coalesce(concat_ws("; ", col("_quarantine_reason"), lit("inspection_date_null")), lit("inspection_date_null")))
    .otherwise(col("_quarantine_reason"))
)
 
# VR-003: inspection_type cannot be null/empty
chicago = chicago.withColumn("_quarantine_reason",
    when(col("inspection_type").isNull() | (trim(col("inspection_type")) == ""),
        coalesce(concat_ws("; ", col("_quarantine_reason"), lit("inspection_type_null")), lit("inspection_type_null")))
    .otherwise(col("_quarantine_reason"))
)
 
# VR-004: zip_code must be valid format (5-digit or ZIP+4)
chicago = chicago.withColumn("_quarantine_reason",
    when(col("zip_code").isNull() | ~col("zip_code").rlike("^\\d{5}(-\\d{4})?$"),
        coalesce(concat_ws("; ", col("_quarantine_reason"), lit("zip_invalid")), lit("zip_invalid")))
    .otherwise(col("_quarantine_reason"))
)
 
# VR-006: Chicago results cannot be null
chicago = chicago.withColumn("_quarantine_reason",
    when(col("results").isNull() | (trim(col("results")) == ""),
        coalesce(concat_ws("; ", col("_quarantine_reason"), lit("results_null")), lit("results_null")))
    .otherwise(col("_quarantine_reason"))
)
 
# VR-009 (Chicago): PASS cannot have Urgent/Critical violations
chicago = chicago.withColumn("_quarantine_reason",
    when((col("results") == "Pass") & (lower(col("violations_raw")).rlike("urgent|critical")),
        coalesce(concat_ws("; ", col("_quarantine_reason"), lit("pass_with_urgent_critical")), lit("pass_with_urgent_critical")))
    .otherwise(col("_quarantine_reason"))
)
 
print("Quarantine reasons tagged")

In [0]:
# Split into clean and quarantine
chicago_quarantine = chicago.filter(col("_quarantine_reason").isNotNull())
chicago_clean = chicago.filter(col("_quarantine_reason").isNull()).drop("_quarantine_reason")
 
chi_passed = chicago_clean.count()
chi_failed = chi_source_count - chi_passed
 
print(f"Chicago DQX: {chi_source_count} total → {chi_passed} passed, {chi_failed} quarantined")

In [0]:
# Write quarantine table
(chicago_quarantine.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{silver_schema}.quarantine_chicago"))
 
print(f"Quarantine table written: {chicago_quarantine.count()} rows")
display(chicago_quarantine.groupBy("_quarantine_reason").count().orderBy(col("count").desc()))

In [0]:
#Write silver.chicago_inspections
chicago_inspections = chicago_clean.select(
    "inspection_id", "dba_name", "aka_name", "license_number",
    "facility_type", "risk", "address", "city", "state", "zip_code",
    "inspection_date", "inspection_type", "results", "violation_score",
    "violations_raw", "latitude", "longitude", "is_out_of_area"
)
 
(chicago_inspections.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{silver_schema}.chicago_inspections"))
 
chi_insp_count = spark.table(f"{catalog_name}.{silver_schema}.chicago_inspections").count()
print(f"silver.chicago_inspections: {chi_insp_count} rows")

In [0]:
#Parse violation blob 
chicago_violations = (
    chicago_clean
    .select("inspection_id", "violations_raw")
    .filter(col("violations_raw").isNotNull() & (trim(col("violations_raw")) != ""))
    .withColumn("violation_item", explode(split(col("violations_raw"), "\\|")))
    .withColumn("violation_item", trim(col("violation_item")))
    .filter(col("violation_item") != "")
    .withColumn("violation_code",
        trim(regexp_extract(col("violation_item"), r"^(\d+)\.", 1)))
    .withColumn("violation_description",
        trim(regexp_extract(col("violation_item"), r"^\d+\.\s*([^-]+?)(?:\s*-\s*Comments:|$)", 1)))
    .withColumn("violation_detail",
        trim(regexp_extract(col("violation_item"), r"-\s*Comments:\s*(.*)$", 1)))
    .withColumn("violation_points", lit(None).cast("double"))
    .withColumn("violation_memo", lit(None).cast("string"))
    .filter(col("violation_code") != "")
    .select("inspection_id", "violation_code", "violation_description",
            "violation_detail", "violation_memo", "violation_points")
)

In [0]:
#deduplicate violation per inspection
chi_viol_before_dedup = chicago_violations.count()
chicago_violations = chicago_violations.dropDuplicates(["inspection_id", "violation_code", "violation_description"])
chi_viol_after_dedup = chicago_violations.count()
 
print(f"Chicago violations: {chi_viol_before_dedup} → {chi_viol_after_dedup} after dedup ({chi_viol_before_dedup - chi_viol_after_dedup} duplicates removed)")

In [0]:
#Handle inspections with zero violations

# Find inspections that had no violations parsed
chi_inspections_with_violations = chicago_violations.select("inspection_id").distinct()
chi_all_inspections = chicago_clean.select("inspection_id")
 
chi_no_violation_ids = chi_all_inspections.join(
    chi_inspections_with_violations, "inspection_id", "left_anti"
)
 
chi_no_viol_count = chi_no_violation_ids.count()
print(f"Inspections with no violations: {chi_no_viol_count}")
 
# Create default rows
chi_default_violations = (chi_no_violation_ids
    .withColumn("violation_code", lit("NO_VIOLATION"))
    .withColumn("violation_description", lit("No violation recorded"))
    .withColumn("violation_detail", lit(None).cast("string"))
    .withColumn("violation_memo", lit(None).cast("string"))
    .withColumn("violation_points", lit(None).cast("double"))
)

# Union with parsed violations
chicago_violations = chicago_violations.unionByName(chi_default_violations)

In [0]:
# Write violations table
(chicago_violations.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{silver_schema}.chicago_violations"))
 
chi_viol_final = spark.table(f"{catalog_name}.{silver_schema}.chicago_violations").count()
print(f"silver.chicago_violations: {chi_viol_final} rows (incl. {chi_no_viol_count} 'No Violation' defaults)")

In [0]:
# Log Chicago DQX
log_dqx("chicago_inspections", "Chicago", chi_source_count, chi_passed, chi_failed)
print(f"Chicago logged — total: {chi_source_count}, passed: {chi_passed}, quarantined: {chi_failed}")